# Bringing Your Own Agent to NeMo Agent Toolkit

In this notebook, we will investigate the integration process of incorporating a pre-existing agent into the NeMo Agent toolkit (NAT) ecosystem.

## Prerequisites

- **Platform:** Linux, macOS, or Windows
- **Python:** version 3.11, 3.12, or 3.13
- **Python Packages:** `pip`

### API Keys

For this notebook, you will need the following API keys to run all examples end-to-end:

- **NVIDIA Build:** You can obtain an NVIDIA Build API Key by creating an [NVIDIA Build](https://build.nvidia.com) account and generating a key at https://build.nvidia.com/settings/api-keys
- **Tavily:** You can obtain a Tavily API Key by creating a [Tavily](https://www.tavily.com/) account and generating a key at https://app.tavily.com/home

Then you can run the cell below:

In [ ]:
import getpass
import os

if "NVIDIA_API_KEY" not in os.environ:
    nvidia_api_key = getpass.getpass("Enter your NVIDIA API key: ")
    os.environ["NVIDIA_API_KEY"] = nvidia_api_key

if "TAVILY_API_KEY" not in os.environ:
    tavily_api_key = getpass.getpass("Enter your Tavily API key: ")
    os.environ["TAVILY_API_KEY"] = tavily_api_key

## Installing NeMo Agent Toolkit

The recommended way to install NAT is through `pip` or `uv pip`.

First, we will install `uv` which offers parallel downloads and faster dependency resolution.

In [ ]:
!pip install uv

NeMo Agent toolkit can be installed through the PyPi `nvidia-nat` package.

There are several optional subpackages available for NAT. The `langchain` subpackage contains useful components for integrating and running within [LangChain](https://python.langchain.com/docs/introduction/). Since LangChain will be used later in this notebook, let's install NAT with the optional `langchain` subpackage.

In [ ]:
%%bash
uv pip install "nvidia-nat[langchain]"

## Your Pre-existing Agent

In this case study, we will have a simple, self-contained LangChain agent capable of searching the internet by using Tavily.

In [ ]:
%%writefile langchain_agent.py
import os

from langchain import hub
from langchain.agents import AgentExecutor
from langchain.agents import create_react_agent
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langchain_tavily import TavilySearch

# Initialize a tool to search the web
search = TavilySearch(
    max_results=2,
    api_key=os.getenv("TAVILY_API_KEY")
)

# Initialize a LLM client
llm = ChatNVIDIA(
    model_name="meta/llama-3.3-70b-instruct",
    temperature=0.0,
    max_completion_tokens=1024,
    api_key=os.getenv("NVIDIA_API_KEY")
)

# Use an open source prompt
prompt = hub.pull("hwchase17/react-chat")

# create tools list
tools = [search]

# Initialize a ReAct agent
react_agent = create_react_agent(
    llm=llm,
    tools=tools,
    prompt=prompt,
    stop_sequence=["\nObservation"]
)

# Initialize an agent executor to iterate through reasoning steps
agent_executor = AgentExecutor(
    agent=react_agent,
    tools=[search],
    max_iterations=15,
    handle_parsing_errors=True,
    verbose=True
)

# Invoke the agent with a user query
response = agent_executor.invoke({"input": "Who won the last World Cup?", "chat_history": []})

# Print the response
print(response["output"])

All of the components in use come from LangGraph/LangChain, but any other example could also work.

Next we will run this sample agent:

In [ ]:
!python langchain_agent.py

There are three main components to this agent:

* a web search tool (Tavily)

* an LLM (Llama 3.3)

* a prompt (obtained from the internet)

The agent is constructed from these three components, then an _agent executor_ is created. Finally, we pass the requested input into the executor and get a response back.

## Migration Part 1: Transforming Your Agent into a Workflow

For the first pass at NAT migration, we will create a new workflow:

In [ ]:
!nat workflow create first_agent_attempt

In [ ]:
%%writefile first_agent_attempt/src/first_agent_attempt/first_agent_attempt.py
import logging

from pydantic import Field

from nat.builder.builder import Builder
from nat.builder.framework_enum import LLMFrameworkEnum
from nat.builder.function_info import FunctionInfo
from nat.cli.register_workflow import register_function
from nat.data_models.function import FunctionBaseConfig

logger = logging.getLogger(__name__)


class FirstAgentAttemptFunctionConfig(FunctionBaseConfig, name="first_agent_attempt"):
    pass


@register_function(config_type=FirstAgentAttemptFunctionConfig, framework_wrappers=[LLMFrameworkEnum.LANGCHAIN])
async def first_agent_attempt_function(_config: FirstAgentAttemptFunctionConfig, _builder: Builder):
    import os

    from langchain import hub
    from langchain.agents import AgentExecutor
    from langchain.agents import create_react_agent
    from langchain_nvidia_ai_endpoints import ChatNVIDIA
    from langchain_tavily import TavilySearch

    # Initialize a tool to search the web
    search = TavilySearch(
        max_results=2,
        api_key=os.getenv("TAVILY_API_KEY")
    )

    # Initialize a LLM client
    llm = ChatNVIDIA(
        model_name="meta/llama-3.3-70b-instruct",
        temperature=0.0,
        max_completion_tokens=1024,
        api_key=os.getenv("NVIDIA_API_KEY")
    )

    # Use an open source prompt
    prompt = hub.pull("hwchase17/react-chat")

    # create tools list
    tools = [search]

    # Initialize a ReAct agent
    react_agent = create_react_agent(
        llm=llm,
        tools=tools,
        prompt=prompt,
        stop_sequence=["\nObservation"]
    )

    # Initialize an agent executor to iterate through reasoning steps
    agent_executor = AgentExecutor(
        agent=react_agent,
        tools=[search],
        max_iterations=15,
        handle_parsing_errors=True,
        verbose=True
    )

    async def _response_fn(input_message: str) -> str:
        response = agent_executor.invoke({"input": input_message, "chat_history": []})

        return response["output"]

    yield FunctionInfo.from_fn(_response_fn, description="A simple tool capable of basic internet search")

This is almost the exact same code indented to fit within a NAT function registration.

The only difference is the definition of a closure function `_response_fn` which captures the instantiated agent executor and uses that to invoke the agent and return the response.

We can also simplify the workflow configuration:

In [ ]:
%%writefile first_agent_attempt/configs/config.yml
workflow:
  _type: first_agent_attempt

Then we can run the new workflow:

In [ ]:
!nat run --config_file first_agent_attempt/configs/config.yml --input "Who won the last World Cup?"

This first pass shows how little effort is required to bring an existing agent into NAT. But we can also extend this further to offer better configuration!

## Migration Part 2: Making Your Agent Configurable

For this next part, we will create another workflow:

In [ ]:
!nat workflow create second_agent_attempt

Then we can update the agent's function.

Below, we expand the configuration to include:

* the LLM it should use
* configurable values for iterations, verbosity, error handling
* an optional description


In [ ]:
%%writefile second_agent_attempt/src/second_agent_attempt/second_agent_attempt.py
import logging

from pydantic import Field

from nat.builder.builder import Builder
from nat.builder.framework_enum import LLMFrameworkEnum
from nat.builder.function_info import FunctionInfo
from nat.cli.register_workflow import register_function
from nat.data_models.component_ref import FunctionRef
from nat.data_models.component_ref import LLMRef
from nat.data_models.function import FunctionBaseConfig

logger = logging.getLogger(__name__)


class SecondAgentAttemptFunctionConfig(FunctionBaseConfig, name="second_agent_attempt"):
    llm_model_name: str = Field(description="LLM name to use")
    max_iterations: int = Field(default=15, description="Maximum number of iterations to run the agent")
    handle_parsing_errors: bool = Field(default=True, description="Whether to handle parsing errors")
    verbose: bool = Field(default=True, description="Whether to print verbose output")
    description: str = Field(default="", description="Description of the agent")


@register_function(config_type=SecondAgentAttemptFunctionConfig, framework_wrappers=[LLMFrameworkEnum.LANGCHAIN])
async def second_agent_attempt_function(config: SecondAgentAttemptFunctionConfig, builder: Builder):
    import os

    from langchain import hub
    from langchain.agents import AgentExecutor
    from langchain.agents import create_react_agent
    from langchain_nvidia_ai_endpoints import ChatNVIDIA
    from langchain_tavily import TavilySearch

    # Initialize a tool to search the web
    search = TavilySearch(
        max_results=2,
        api_key=os.getenv("TAVILY_API_KEY")
    )

    # Initialize a LLM client
    llm = ChatNVIDIA(
        model_name=config.llm_model_name,
        temperature=0.0,
        max_completion_tokens=1024,
        api_key=os.getenv("NVIDIA_API_KEY")
    )

    # Use an open source prompt
    prompt = hub.pull("hwchase17/react-chat")

    # create tools list
    tools = [search]

    # Initialize a ReAct agent
    react_agent = create_react_agent(
        llm=llm,
        tools=tools,
        prompt=prompt,
        stop_sequence=["\nObservation"]
    )

    # Initialize an agent executor to iterate through reasoning steps
    agent_executor = AgentExecutor(
        agent=react_agent,
        tools=[search],
        **config.model_dump(include={"max_iterations", "handle_parsing_errors", "verbose"})
    )

    async def _response_fn(input_message: str) -> str:
        response = agent_executor.invoke({"input": input_message, "chat_history": []})

        return response["output"]

    yield FunctionInfo.from_fn(_response_fn, description=config.description)

We can then update the configuration file to include the configuration options which previously were embedded into the agent's code:

In [ ]:
%%writefile second_agent_attempt/configs/config.yml
workflow:
  _type: second_agent_attempt
  llm_model_name: meta/llama-3.3-70b-instruct
  max_iterations: 15
  verbose: false
  description: "A helpful assistant that can search the internet for information"

We can then run this modified agent to demonstrate the YAML configuration capabilities of NeMo Agent toolkit.

In [ ]:
!nat run --config_file second_agent_attempt/configs/config.yml --input "Who won the last World Cup?"

## Migration Part 3: Integration with NeMo Agent Toolkit

NeMo Agent toolkit comes with support for various LLM Providers, Frameworks, and additional components.

For this last part of migrating an agent, we will adapt the agent to use built-in toolkit components rather than importing directly from LangChain.

Changes made below:
- changing from LLM model name to an LLM _reference_
- adapting the code to query NAT for the LLM and Tools to use
- switching to the built-in Tavily Search Tool

In [ ]:
!nat workflow create third_agent_attempt

In [ ]:
%%writefile third_agent_attempt/src/third_agent_attempt/third_agent_attempt.py
import logging

from pydantic import Field

from nat.builder.builder import Builder
from nat.builder.framework_enum import LLMFrameworkEnum
from nat.builder.function_info import FunctionInfo
from nat.cli.register_workflow import register_function
from nat.data_models.component_ref import FunctionRef
from nat.data_models.component_ref import LLMRef
from nat.data_models.function import FunctionBaseConfig

logger = logging.getLogger(__name__)


class ThirdAgentAttemptFunctionConfig(FunctionBaseConfig, name="third_agent_attempt"):
    tool_names: list[FunctionRef] = Field(defaultfactory=list, description="List of tool names to use")
    llm_name: LLMRef = Field(description="LLM name to use")
    max_iterations: int = Field(default=15, description="Maximum number of iterations to run the agent")
    handle_parsing_errors: bool = Field(default=True, description="Whether to handle parsing errors")
    verbose: bool = Field(default=True, description="Whether to print verbose output")
    description: str = Field(default="", description="Description of the agent")

# Since our agent relies on Langchain, we must explicitly list the supported framework wrappers.
# Otherwise, the toolkit would not know the correct type to return from the builder

@register_function(config_type=ThirdAgentAttemptFunctionConfig, framework_wrappers=[LLMFrameworkEnum.LANGCHAIN])
async def third_agent_attempt_function(config: ThirdAgentAttemptFunctionConfig, builder: Builder):
    import os

    from langchain import hub
    from langchain.agents import AgentExecutor
    from langchain.agents import create_react_agent

    # Create a list of tools for the agent
    tools = await builder.get_tools(config.tool_names, wrapper_type=LLMFrameworkEnum.LANGCHAIN)

    llm = await builder.get_llm(config.llm_name, wrapper_type=LLMFrameworkEnum.LANGCHAIN)

    # Use an open source prompt
    prompt = hub.pull("hwchase17/react-chat")

    # Initialize a ReAct agent
    react_agent = create_react_agent(
        llm=llm,
        tools=tools,
        prompt=prompt,
        stop_sequence=["\nObservation"]
    )

    # Initialize an agent executor to iterate through reasoning steps
    agent_executor = AgentExecutor(
        agent=react_agent,
        tools=tools,
        **config.model_dump(include={"max_iterations", "handle_parsing_errors", "verbose"})
    )

    async def _response_fn(input_message: str) -> str:
        response = agent_executor.invoke({"input": input_message, "chat_history": []})

        return response["output"]

    yield FunctionInfo.from_fn(_response_fn)

We can then update the configuration file to include LLM and Function definitions that before were embedded into the agent's code:

In [ ]:
%%writefile third_agent_attempt/configs/config.yml
llms:
  nim_llm:
    _type: nim
    model_name: meta/llama-3.3-70b-instruct
    temperature: 0.0
    max_tokens: 1024
    api_key: $NVIDIA_API_KEY

functions:
  search:
    _type: tavily_internet_search
    max_results: 2
    api_key: $TAVILY_API_KEY

workflow:
  _type: third_agent_attempt
  tool_names: [search]
  llm_name: nim_llm
  max_iterations: 15
  verbose: false
  description: "A helpful assistant that can search the internet for information"

Finally, we can run this modified agent to demonstrate the flexibility and adaptiveness of using NeMo Agent toolkit.

In [ ]:
!nat run --config_file third_agent_attempt/configs/config.yml --input "Who won the last World Cup?"

## Migration Part 4: A Zero-Code Configuration

Sometimes NeMo Agent toolkit has all of the components you need already. In cases like these, we can rely on zero code additions. The effect of this is being able to **only** specify a configuration file, demonstrating the power of a batteries-included approach.

The required components for this base example were:
- An LLM (NVIDIA NIM-based)
- Tavily Internet Search Tool
- ReAct Agent

In [ ]:
%%writefile search_agent.yml
llms:
  nim_llm:
    _type: nim
    model_name: meta/llama-3.3-70b-instruct
    temperature: 0.0
    max_tokens: 1024
    api_key: $NVIDIA_API_KEY

functions:
  search:
    _type: tavily_internet_search
    max_results: 2
    api_key: $TAVILY_API_KEY

workflow:
  _type: react_agent
  tool_names: [search]
  llm_name: nim_llm
  verbose: false
  description: "A helpful assistant that can search the internet for information"

In [ ]:
!nat run --config_file search_agent.yml --input "Who won the last World Cup?"

This concludes the "Bringing Your Own Agent to NeMo Agent toolkit" notebook.



## Next Steps

Next, look at "Adding Tools and Agents to NeMo Agent Toolkit" where you will interactively learn how to create your own tools and agents with NAT.